In [ ]:
import os, sys
os.chdir('../..')
import numpy as np
from benchmarks_august.targets.logreg import logreg_synthetic
from benchmarks_august.samplers import build_sampler, apply_preprocess
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path
from benchmarks_august.analysis.metrics import logreg_performance
from benchmarks_august.samplers.warmstart import warmup_reference
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
# Shared settings
N = 10000
n_resample = 50000
burnin = 0.2
refresh_rate = 1.0

In [ ]:
target_dense = logreg_synthetic(n=500, p=10, sparsity='dense', seed=42,
                                prior={'kind': 'gaussian', 'scale': 1.0})
print('beta_true:', target_dense.true_params)

X_d, y_d = target_dense.data['X'], target_dense.data['y']
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_d, y_d, test_size=0.2, random_state=0)

# Rebuild target on train split for fair evaluation
from benchmarks_august.targets.logreg import logreg
target_dense_train = logreg(X_tr_d, y_tr_d, prior={'kind': 'gaussian', 'scale': 1.0})

# sklearn reference
lr_d = LogisticRegression(C=1.0, penalty='l2', fit_intercept=False, max_iter=1000).fit(X_tr_d, y_tr_d)
print('sklearn MAP:', lr_d.coef_[0])

In [ ]:
# --- Boomerang PLI (dense) ---
boom_d_raw = build_sampler('boomerang', target_dense_train, N=N, refresh_rate=refresh_rate)
apply_preprocess(boom_d_raw, target_dense_train, {'method': 'diagonal'})
boom_d_raw.sample_auto(diagnostics=True)


In [ ]:
# --- Boomerang PLI (dense) ---
boom_d_adapt = build_sampler('boomerang', target_dense_train, N=N, refresh_rate=refresh_rate)
warmup_reference(boom_d_adapt, n_rounds=3, n_pilot=500)
boom_d_adapt.reset(N=N)
boom_d_adapt.sample_auto(diagnostics=True)

In [ ]:
# Resample & visualize
_, x_boom_raw = resample_pdmp_path(boom_d_raw, n_samples=n_resample, burnin_frac=burnin)
_, x_boom_adapt = resample_pdmp_path(boom_d_adapt, n_samples=n_resample, burnin_frac=burnin)

In [ ]:
# Performance
logreg_performance(X_tr_d, y_tr_d, X_te_d, y_te_d, {
    'Boomerang raw': x_boom_raw,
    'Boomerang adaptive': x_boom_adapt,
}, burnin_frac=0)  # already burned in

In [ ]:
boom_d_raw.Sigma[:5, :5]

In [ ]:
boom_d_adapt.Sigma[:5, :5]